Download the subset dataset from the Google Drive link in https://github.com/VideoObjectSearch/RandPerson
Extract images from it and put in the directory structure as shown below, or modify the next code to fit your directory structure
```
randperson_dataset
└── randperson_subset
    └── crop_images
        ├── 000000_s00_c00_f000228.jpg
        ├── 000000_s01_c00_f000144.jpg
        ├── 000000_s01_c01_f000300.jpg
        ├── 000000_s02_c00_f000684.jpg
        ......
```
Run the following cells to perform the finetuning. No query test set was prepared, the weights were directly saved after training.

In [ ]:
import os
import glob
import re
from torchreid.reid.data import ImageDataset

class RandPersonSubset(ImageDataset):
    dataset_dir = 'randperson_subset'

    def __init__(self, root='', **kwargs):
        # Normalize the root path for Windows compatibility
        self.root = os.path.normpath(os.path.abspath(root))
        self.dataset_dir = os.path.join(self.root, self.dataset_dir)
        
        # Ensure 'crop_images' folder path is completely accurate
        self.train_dir = os.path.join(self.dataset_dir, 'crop_images')
        
        # Debugging print statement: run your script and verify this printed path is real!
        print(f"==> Searching for crop images in: {self.train_dir}")
        
        train = self.process_dir(self.train_dir, relabel=True)
        
        # Sanity check to provide a helpful error message instead of an IndexError crash
        if len(train) == 0:
            raise ValueError(f"No images found in {self.train_dir}.")

        # Look for at least two different images belonging to the same person
        # train[i] structure is: (img_path, pid, camid)
        mock_query = []
        mock_gallery = []

        # Simple logic to find a matching ID with different images
        if len(train) >= 2:
            mock_query = [train[0]]    # First image of Person A
            
            # Try to find a different image of Person A for the gallery
            found_match = False
            for item in train[1:]:
                if item[1] == train[0][1]: # Same pid, different path
                    mock_gallery = [item]
                    found_match = True
                    break
                    
            # Fallback if no matching IDs are found (just to prevent crashes)
            if not found_match:
                mock_gallery = [train[1]] 
        else:
            mock_query = train
            mock_gallery = train

        # Pass the mock sets to the super constructor
        super(RandPersonSubset, self).__init__(train, mock_query, mock_gallery, **kwargs)

    def process_dir(self, dir_path, relabel=False):
        img_paths = glob.glob(os.path.join(dir_path, '*.jpg'))
        
        # Regex to parse: [PersonID]_s[SeqID]_c[CamID]_f[FrameID].jpg
        # Example: 000002_s00_c00_f000048.jpg
        # Match Group 1: 000002 (Person ID), Match Group 2: 00 (Camera ID)
        pattern = re.compile(r'([\d]+)_s[\d]+_c([\d]+)_f[\d]+')

        pid_container = set()
        for img_path in img_paths:
            filename = os.path.basename(img_path)
            match = pattern.search(filename)
            if match:
                pid, _ = map(int, match.groups())
                pid_container.add(pid)
                
        # Map arbitrary person IDs to a continuous range starting at 0 (required by Cross-Entropy loss)
        pid2label = {pid: label for label, pid in enumerate(sorted(pid_container))}

        data = []
        for img_path in img_paths:
            filename = os.path.basename(img_path)
            match = pattern.search(filename)
            if not match:
                print(f"Filename {filename} does not match expected pattern. Skipping.")
                continue # Skip files that don't match the expected structure
                
            pid, camid = map(int, match.groups())
            
            if relabel:
                pid = pid2label[pid]
            
            data.append((img_path, pid, camid))

        return data

In [ ]:
import torchreid

# 1. Register the custom dataset
torchreid.data.register_image_dataset('randperson', RandPersonSubset)

# 2. Setup the DataManager
datamanager = torchreid.data.ImageDataManager(
    root='randperson_dataset',
    sources='randperson',
    targets='randperson', 
    height=256,
    width=128,
    batch_size_train=128,
    workers=8,
)

Building train transforms ...
+ resize to 256x128
+ random flip
+ to torch tensor of range [0, 1]
+ normalization (mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
Building test transforms ...
+ resize to 256x128
+ to torch tensor of range [0, 1]
+ normalization (mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
=> Loading train (source) dataset
==> Searching for crop images in: /home/user/pythonprojects/randperson/randperson_dataset/randperson_subset/crop_images
=> Loaded RandPersonSubset
  ----------------------------------------
  subset   | # ids | # images | # cameras
  ----------------------------------------
  train    |  8000 |   132145 |         4
  query    |     1 |        1 |         1
  gallery  |     1 |        1 |         1
  ----------------------------------------
=> Loading test (target) dataset
==> Searching for crop images in: /home/user/pythonprojects/randperson/randperson_dataset/randperson_subset/crop_images
=> Loaded RandPersonSubset
  ---------------

In [ ]:
# Fix for PyTorch 2.6 or above

import pickle
from functools import partial
from collections import OrderedDict
import torch
import os.path as osp

def load_checkpoint(fpath):
    r"""Loads checkpoint.

    ``UnicodeDecodeError`` can be well handled, which means
    python2-saved files can be read from python3.

    Args:
        fpath (str): path to checkpoint.

    Returns:
        dict

    Examples::
        >>> from torchreid.utils import load_checkpoint
        >>> fpath = 'log/my_model/model.pth.tar-10'
        >>> checkpoint = load_checkpoint(fpath)
    """
    if fpath is None:
        raise ValueError('File path is None')
    fpath = osp.abspath(osp.expanduser(fpath))
    if not osp.exists(fpath):
        raise FileNotFoundError('File is not found at "{}"'.format(fpath))
    map_location = None if torch.cuda.is_available() else 'cpu'
    try:
        checkpoint = torch.load(fpath, map_location=map_location, weights_only=False)
    except UnicodeDecodeError:
        pickle.load = partial(pickle.load, encoding="latin1")
        pickle.Unpickler = partial(pickle.Unpickler, encoding="latin1")
        checkpoint = torch.load(
            fpath, pickle_module=pickle, map_location=map_location,weights_only=False
        )
    except Exception:
        print('Unable to load checkpoint from "{}"'.format(fpath))
        raise
    return checkpoint

def load_pretrained_weights(model, weight_path):
    r"""Loads pretrianed weights to model.

    Features::
        - Incompatible layers (unmatched in name or size) will be ignored.
        - Can automatically deal with keys containing "module.".

    Args:
        model (nn.Module): network model.
        weight_path (str): path to pretrained weights.

    Examples::
        >>> from torchreid.utils import load_pretrained_weights
        >>> weight_path = 'log/my_model/model-best.pth.tar'
        >>> load_pretrained_weights(model, weight_path)
    """
    checkpoint = load_checkpoint(weight_path)
    if 'state_dict' in checkpoint:
        state_dict = checkpoint['state_dict']
    else:
        state_dict = checkpoint

    model_dict = model.state_dict()
    new_state_dict = OrderedDict()
    matched_layers, discarded_layers = [], []

    for k, v in state_dict.items():
        if k.startswith('module.'):
            k = k[7:] # discard module.

        if k in model_dict and model_dict[k].size() == v.size():
            new_state_dict[k] = v
            matched_layers.append(k)
        else:
            discarded_layers.append(k)

    model_dict.update(new_state_dict)
    model.load_state_dict(model_dict)

    if len(matched_layers) == 0:
        print.warn(
            'The pretrained weights "{}" cannot be loaded, '
            'please check the key names manually '
            '(** ignored and continue **)'.format(weight_path)
        )
    else:
        print(
            'Successfully loaded pretrained weights from "{}"'.
            format(weight_path)
        )
        if len(discarded_layers) > 0:
            print(
                '** The following layers are discarded '
                'due to unmatched keys or layer size: {}'.
                format(discarded_layers)
            )

In [5]:
import torch
# ==========================================
# 3. BUILD MODEL & CONFIGURE TRAINING
# ==========================================
# Build the OSNet architecture (dynamically matching your subset's unique person count)
model = torchreid.models.build_model(
    name='osnet_x1_0',
    num_classes=datamanager.num_train_pids,
    loss='softmax',
    pretrained=False  # Warm start using ImageNet weights
)
load_pretrained_weights(model, './weights/osnet_ms_d_c.pth.tar')

if torch.cuda.is_available():
    model = model.cuda()

# Set up optimized Adam hyperparameters standard for OSNet training
optimizer = torchreid.optim.build_optimizer(
    model,
    optim='adam',
    lr=0.0003
)

scheduler = torchreid.optim.build_lr_scheduler(
    optimizer,
    lr_scheduler='single_step',
    stepsize=20,
    gamma=0.1
)

# Initialize the training loop wrapper
engine = torchreid.engine.ImageSoftmaxEngine(
    datamanager,
    model,
    optimizer=optimizer,
    scheduler=scheduler,
    label_smooth=True
)


Successfully loaded pretrained weights from "./weights/osnet_ms_d_c.pth.tar"
** The following layers are discarded due to unmatched keys or layer size: ['classifier.weight', 'classifier.bias']


In [7]:
# ==========================================
# 4. EXECUTE TRAINING
# ==========================================
engine.run(
    max_epoch=30,
    save_dir='log/osnet_randperson',
    print_freq=10,
    test_only=False,  # Tells engine to run training epochs, not just test cycles
    start_eval=999,      # Prevents evaluation during training loops
    ranks=[1]
)

=> Start training
epoch: [1/30][10/1032]	time 0.319 (0.518)	data 0.001 (0.089)	eta 4:27:21	loss 8.9984 (9.0245)	acc 0.7812 (0.1562)	lr 0.000300
epoch: [1/30][20/1032]	time 0.298 (0.418)	data 0.001 (0.045)	eta 3:35:24	loss 9.0802 (9.0398)	acc 0.7812 (0.1562)	lr 0.000300
epoch: [1/30][30/1032]	time 0.318 (0.382)	data 0.001 (0.030)	eta 3:16:47	loss 9.1291 (9.0763)	acc 0.7812 (0.1562)	lr 0.000300
epoch: [1/30][40/1032]	time 0.316 (0.365)	data 0.001 (0.023)	eta 3:08:16	loss 9.1610 (9.1107)	acc 0.0000 (0.1758)	lr 0.000300
epoch: [1/30][50/1032]	time 0.288 (0.355)	data 0.001 (0.018)	eta 3:02:57	loss 9.3038 (9.1339)	acc 0.0000 (0.1875)	lr 0.000300
epoch: [1/30][60/1032]	time 0.318 (0.347)	data 0.001 (0.015)	eta 2:58:40	loss 9.0628 (9.1343)	acc 0.0000 (0.2214)	lr 0.000300
epoch: [1/30][70/1032]	time 0.317 (0.343)	data 0.001 (0.013)	eta 2:56:26	loss 8.8884 (9.1166)	acc 0.7812 (0.3348)	lr 0.000300
epoch: [1/30][80/1032]	time 0.316 (0.339)	data 0.001 (0.012)	eta 2:54:39	loss 8.7687 (9.0891)	acc 0.

In [ ]:
torch.save(model.state_dict(), 'log/osnet_randperson/osnet_randperson_ms_d_c.pth')